In [6]:
import pandas as pd
import numpy as np

# **Crossmatch Data with Gaia DR3**

Read out ID, RA, and Dec for cleaned combined catalogue.

In [7]:
full_data = pd.read_csv("../2_duplicatestars/asteroseismic_clean.csv")

idents = pd.DataFrame({
    "gdr3_id": full_data["gaiadr3_id"],
    "ra": full_data["ra"],
    "dec": full_data["dec"]
})

# write to .csv file
idents.to_csv("../0_inputdata_spectroscopic/idents.csv", index=False)

## **Xmatch with Gaia Params + BJ21 Distances + Extinction Map**

Paper : https://ui.adsabs.harvard.edu/abs/2021AJ....161..147B/abstract

Catalogue : https://ui.adsabs.harvard.edu/abs/2021yCat.1352....0B/abstract

ADQL Query (substituting $\frac{2.5}{\ln(10)} \approx 1.0857362047581296$):

```
SELECT 
    lj.gdr3_id AS gdr3_id,
    lj.ra AS ra,
    lj.dec AS dec,

    bj.r_med_geo AS bj_rgeo,
    bj.r_lo_geo AS bj_b_rgeo,
    bj.r_hi_geo AS bj_B_rgeo,

    bj.r_med_photogeo AS bj_rphoto,
    bj.r_lo_photogeo AS bj_b_rphoto,
    bj.r_hi_photogeo AS bj_B_rphoto,
    bj.flag AS bj_flag,

    g.phot_g_mean_mag AS gaia_gmag,
    1.0857362047581296 / g.phot_g_mean_flux_over_error AS gaia_e_gmag,

    ex.a0 AS map_a0,
    ex.a0_uncertainty AS map_e_a0,

    g.ruwe AS gaia_ruwe,
    g.pmra AS gaia_pmra,
    g.pmra_error AS gaia_e_pmra,
    g.pmdec AS gaia_pmdec,
    g.pmdec_error AS gaia_e_pmdec,

    g.parallax AS gaia_plx,
    g.parallax_error AS gaia_e_plx,
    g.radial_velocity AS gaia_rv,
    g.radial_velocity_error AS gaia_e_rv

FROM user_lspriggs.idents AS lj

LEFT JOIN external.gaiaedr3_distance AS bj
    ON lj.gdr3_id = bj.source_id

LEFT JOIN gaiadr3.gaia_source AS g
    ON lj.gdr3_id = g.source_id

LEFT JOIN gaiadr3.total_galactic_extinction_map_opt AS ex
    ON ex.healpix_id = GAIA_HEALPIX_INDEX(9, g.source_id)
```

Read-in and merge data.

In [8]:
ext_data = pd.read_csv("../0_inputdata_spectroscopic/gaia_params.csv")

# rename columns
ext_data.columns = ext_data.columns.str.replace("bj_", "BJ-")
ext_data.columns = ext_data.columns.str.replace("gaia_", "GAIA-")
ext_data.columns = ext_data.columns.str.replace("gspphot_", "GSPPHOT-")
ext_data.columns = ext_data.columns.str.replace("map_", "")

# convert lower + upper bounds to single uncertainty
upper_cols = [col for col in ext_data.columns if "-B_" in col] # find all params w upper bound
for upper_col in upper_cols:
    lower_col = upper_col.replace("-B_", "-b_") # find corresponding lower bound
    ext_data[upper_col.replace("-B_", "-e_")] = (ext_data[upper_col] - ext_data[lower_col]) / 2 # compute err value
    ext_data = ext_data.drop(columns=[upper_col, lower_col]) # discard lower + upper bounds

ext_data = ext_data.drop(columns=["ra", "dec"])

int_data = pd.read_csv("../2_duplicatestars/asteroseismic_clean.csv")

data = pd.merge(int_data, ext_data, how="left", left_on="gaiadr3_id", right_on="gdr3_id")

## **Compute Distances from Parallax (if necessary)**

In [9]:
valid_plx_mask = np.isfinite(data["GAIA-plx"]) & (data["GAIA-plx"] > 0) & np.isfinite(data["GAIA-e_plx"]) & np.isfinite(data["GAIA-e_plx"])

# dist = 1 / parallax
plx_dist = np.divide(
    1.0, 
    data["GAIA-plx"], 
    out=np.full(len(data), np.nan, dtype=float), 
    where=valid_plx_mask
)
e_plx_dist = data["GAIA-e_plx"] * (plx_dist**2) # from analytic uncertainties propagation

conditions = [
    np.isfinite(data["BJ-rphoto"]),
    np.isfinite(data["BJ-rgeo"]),
    valid_plx_mask & (plx_dist <= 3e4)
]
choices = [
    data["BJ-rphoto"],
    data["BJ-rgeo"],
    plx_dist
]
e_choices = [
    data["BJ-e_rphoto"],
    data["BJ-e_rgeo"],
    e_plx_dist
]

data["dist"] = np.select(conditions, choices, default=np.nan)
data["e_dist"] = np.select(conditions, e_choices, default=np.nan)

In [10]:
data = data.dropna(subset=["dist", "e_dist", "GAIA-pmra", "GAIA-e_pmra", "GAIA-pmdec", "GAIA-e_pmdec"])

data.to_csv("./gaia_asteroseismic.csv", index=False)

**Account for extinction:**

$$m_{G,\,\mathrm{corr}} = m_{G,\,\mathrm{obs}} - A_G.$$

**Derive absolute magnitude:**

$$M_G = m_G - 5\log_{10}\left(\frac{d}{10\,\mathrm{pc}}\right).$$

**Bolometric correction:**

$$M_{\mathrm{bol}} = M_G + \mathrm{BC}_G.$$

**Compute Luminosity:**

$$L = 10^{0.4\left(M_{\odot,\,\mathrm{bol}} - M_{\mathrm{bol}}\right)}L_\odot.$$